# Notebook 2: Data Preprocessing
## Carbon Footprint Prediction System — BDA5011

This notebook constructs the feature vectors expected by the models as input:
1. **Cleaning**: Missing value handling, outlier detection
2. **Feature Engineering**: Lag features, moving averages, seasonal encoding
3. **Normalization**: Min-Max / Standard Scaler application
4. **Splitting**: Sequential splitting for time series (shuffle=False)

In [ ]:
# =============================================================================
# Library Imports
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import warnings; warnings.filterwarnings('ignore')

plt.style.use('dark_background')
plt.rcParams['axes.facecolor'] = '#16213e'
plt.rcParams['figure.facecolor'] = '#1a1a2e'

DATA_DIR = Path('../data')
print('✓ Ready')

## 1. Missing Value Analysis

In [ ]:
# =============================================================================
# Missing Value Analysis
# =============================================================================
# Determines missing value tolerance throughout the pipeline.
# No missing values are expected in Carbon Monitor (daily predictions are complete).
# In EDGAR, data might be missing for some years.

cm = pd.read_csv(DATA_DIR / 'carbon_monitor' / 'carbon_monitor_global.csv')
edgar = pd.read_csv(DATA_DIR / 'edgar' / 'edgar_country_sector_1990_2023.csv')

datasets = {'Carbon Monitor': cm, 'EDGAR': edgar}

for name, df in datasets.items():
    missing = df.isnull().sum()
    missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
    
    print(f'\n{name} — Missing Value Report:')
    print(f'  Total records: {len(df):,}')
    print(f'  Total missing: {missing.sum()}')
    
    if missing.sum() > 0:
        summary = pd.DataFrame({'Missing': missing[missing > 0], 'Percentage (%)': missing_pct[missing > 0]})
        print(summary)
    else:
        print('  ✓ No missing values')

## 2. Outlier Detection (IQR Method)

In [ ]:
# =============================================================================
# Outlier Detection — IQR (Interquartile Range) Method
# =============================================================================
# IQR: Q3 - Q1 = width of the middle 50% of values
# Lower bound: Q1 - 1.5 * IQR
# Upper bound: Q3 + 1.5 * IQR
# Values outside these bounds are flagged as 'outliers'.

emission_col = 'MtCO2 per day'

Q1  = cm[emission_col].quantile(0.25)
Q3  = cm[emission_col].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = cm[(cm[emission_col] < lower) | (cm[emission_col] > upper)]

print(f'Emission distribution:')
print(f'  Q1: {Q1:.4f} | Q3: {Q3:.4f} | IQR: {IQR:.4f}')
print(f'  Lower bound: {lower:.4f} | Upper bound: {upper:.4f}')
print(f'  Number of outliers: {len(outliers):,} ({len(outliers)/len(cm)*100:.2f}%)')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#1a1a2e')

# Box plot
axes[0].set_facecolor('#16213e')
bp = axes[0].boxplot(cm.groupby('country')[emission_col].apply(list).values,
                     labels=cm['country'].unique(),
                     patch_artist=True,
                     boxprops=dict(facecolor='#00d4ff', alpha=0.6),
                     medianprops=dict(color='white', linewidth=2),
                     whiskerprops=dict(color='#8bafc5'),
                     capprops=dict(color='#8bafc5'),
                     flierprops=dict(marker='o', markerfacecolor='#ff5252', markersize=3))
axes[0].set_title('Emission Distribution by Country (Box Plot)', fontweight='bold')
axes[0].set_ylabel('MtCO₂/day')
axes[0].tick_params(axis='x', rotation=45)

# Histogram
axes[1].set_facecolor('#16213e')
axes[1].hist(cm[emission_col], bins=80, color='#00d4ff', alpha=0.7, edgecolor='none')
axes[1].axvline(x=upper, color='#ff5252', linestyle='--', linewidth=2, label=f'Upper bound ({upper:.2f})')
axes[1].axvline(x=lower, color='#ff9800', linestyle='--', linewidth=2, label=f'Lower bound ({lower:.2f})')
axes[1].set_title('Emission Distribution and IQR Bounds', fontweight='bold')
axes[1].set_xlabel('MtCO₂/day')
axes[1].legend(facecolor='#1a1a2e', labelcolor='white')

plt.tight_layout()
plt.savefig('../results/outlier_analysis.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

## 3. Feature Engineering — Time Series Features

In [ ]:
# =============================================================================
# Feature Engineering — Lag Features and Moving Averages
# =============================================================================
# WHY LAG FEATURES?
#   In time series, today's emission is highly dependent on yesterday's emission (autocorrelation).
#   Lag(1d): Previous day  → short-term dynamic
#   Lag(7d): Last week → weekly seasonality
#   Lag(30d): Last month   → monthly seasonality
#
# WHY MOVING AVERAGE?
#   Reduces noise, captures overall trend.
#   MA(7d): Short-term trend
#   MA(30d): Long-term trend

cm['date'] = pd.to_datetime(cm['date'])
cm = cm.sort_values(['country', 'sector', 'date'])

# Calculate lag and MA per group (GROUP BY country, sector)
grp = cm.groupby(['country', 'sector'])['MtCO2 per day']

cm['lag_1d']         = grp.shift(1)                             # One day ago
cm['lag_7d']         = grp.shift(7)                             # One week ago
cm['lag_30d']        = grp.shift(30)                            # One month ago
cm['rolling_7d_avg'] = grp.transform(lambda x: x.rolling(7, min_periods=1).mean())
cm['rolling_30d_avg']= grp.transform(lambda x: x.rolling(30, min_periods=1).mean())

# Seasonal encoding — sin/cos transformation
# Why sin/cos? So the transition from Month 12 → Month 1 is continuous
cm['year']      = cm['date'].dt.year
cm['month']     = cm['date'].dt.month
cm['month_sin'] = np.sin(2 * np.pi * cm['month'] / 12)
cm['month_cos'] = np.cos(2 * np.pi * cm['month'] / 12)
cm['dow']       = cm['date'].dt.dayofweek   # 0=Mon, 6=Sun
cm['dow_sin']   = np.sin(2 * np.pi * cm['dow'] / 7)
cm['is_weekend']= (cm['dow'] >= 5).astype(int)

# Drop missing values (first records will be NaN due to lag)
cm_clean = cm.dropna(subset=['lag_7d'])

print(f'After feature engineering:')
print(f'  Original: {len(cm):,} → Clean: {len(cm_clean):,} ({len(cm)-len(cm_clean):,} rows dropped)')
print(f'\nCreated features:')
new_features = ['lag_1d', 'lag_7d', 'lag_30d', 'rolling_7d_avg', 'rolling_30d_avg',
                'month_sin', 'month_cos', 'dow_sin', 'is_weekend']
for f in new_features:
    print(f'  {f}: min={cm_clean[f].min():.4f}, max={cm_clean[f].max():.4f}, mean={cm_clean[f].mean():.4f}')

In [ ]:
# =============================================================================
# Time-Based Train/Test Split
# =============================================================================
# WHY TIME-BASED SPLITTING?
#   Random splitting (shuffle=True) causes data leakage!
#   Future data would be used as input to predict past data.
#   Correct approach: first 80% train, last 20% test.

# Display for a single country-sector combination
sample = cm_clean[(cm_clean['country'] == 'CN') & (cm_clean['sector'] == 'Power')].copy()

split_idx = int(len(sample) * 0.80)
train = sample.iloc[:split_idx]
test  = sample.iloc[split_idx:]

print(f'CN/Power data split:')
print(f'  Train: {len(train)} records ({train.date.min().date()} → {train.date.max().date()})')
print(f'  Test:  {len(test)}  records ({test.date.min().date()} → {test.date.max().date()})')

# Visualize
fig, ax = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#16213e')

ax.plot(train['date'], train['MtCO2 per day'], color='#00d4ff', linewidth=1.5, label='Train set')
ax.plot(test['date'],  test['MtCO2 per day'],  color='#ff9800', linewidth=1.5, label='Test set')
ax.axvline(x=train['date'].max(), color='white', linestyle='--', linewidth=2, alpha=0.6)
ax.text(train['date'].max(), ax.get_ylim()[1]*0.9, ' Split\n Point', color='white', fontsize=10)

ax.set_title('CN/Power — Time-Based Train/Test Split (80/20)', fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('MtCO₂/day')
ax.legend(facecolor='#1a1a2e', labelcolor='white')
ax.grid(alpha=0.1)

plt.tight_layout()
plt.savefig('../results/train_test_split.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

In [ ]:
# =============================================================================
# Normalization — StandardScaler
# =============================================================================
# WHY NORMALIZATION?
#   Features with different scales (0-3 MtCO2 vs 2020-2024 years)
#   slow down gradient descent.
#   StandardScaler: (X - mean) / std → mean=0, std=1
#
# IMPORTANT: Fit only on the training set! Use transform for testing.
#   Including the test set would cause data leakage.

feature_cols = ['lag_1d', 'lag_7d', 'lag_30d', 'rolling_7d_avg', 'rolling_30d_avg',
                'month_sin', 'month_cos', 'dow_sin', 'is_weekend', 'year']
target_col   = 'MtCO2 per day'

X_train = train[feature_cols].values
X_test  = test[feature_cols].values
y_train = train[target_col].values
y_test  = test[target_col].values

# Fit the scaler only on train
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)  # SADECE transform for test!

print('Statistics after normalization (Train):')
for i, col in enumerate(feature_cols):
    print(f'  {col:25s}: mean={X_train_scaled[:, i].mean():.4f}, std={X_train_scaled[:, i].std():.4f}')

print(f'\nTarget value range: [{y_train.min():.4f}, {y_train.max():.4f}] MtCO₂/day')
print('\n✓ Data is ready for model training!')